# HUF Triad — Vol 0, Notebook 2
# 🎒 My Backpack: Portfolio Thinking for Everything

**Audience**: Anyone who finished Notebook 1 (Hello Ratios)

**What you will learn**:
- How to model *any* collection as a portfolio
- What **leverage** tells you about each item's importance
- How removing one item affects everything else (the **simplex** in action)
- The **PROOF line** — how concentrated is your portfolio?

**Time**: ~15 minutes

---

> *Previous*: `00_hello_ratios.ipynb` (Pizza and the Unity Constraint)  
> *Next*: `02_sourdough_baker.ipynb` (Real Data: Sourdough Fermentation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
print('Ready! Let\'s pack a backpack.')

## 1. Your Backpack is a Portfolio

A school backpack has a fixed capacity (weight limit). Everything inside shares that capacity.  
Sound familiar? That's a **budget ceiling** with **elements** holding **shares**.

Let's model it.

In [ ]:
# A backpack with a 5 kg budget ceiling
items = ['Books', 'Lunch', 'Water Bottle', 'Pencil Case', 'Jacket']
weights_kg = [2.0, 1.0, 0.5, 0.3, 1.2]  # kg per item

total = sum(weights_kg)
shares = [w / total for w in weights_kg]
leverage = [1 / s for s in shares]

print(f'Budget ceiling: {total:.1f} kg\n')
print(f'{"Item":<15} {"Weight (kg)":<12} {"Share (ρ)":<10} {"Leverage":<10}')
print('-' * 50)
for item, w, s, l in zip(items, weights_kg, shares, leverage):
    print(f'{item:<15} {w:<12.1f} {s:<10.3f} {l:<10.1f}')
print(f'\n\u2211\u03c1 = {sum(shares):.3f}  \u2714 Unity constraint holds.')
print(f'\nHighest leverage: {items[leverage.index(max(leverage))]} (leverage = {max(leverage):.1f})')
print('  \u2192 This item is smallest but most sensitive to removal!')

In [ ]:
# Visualize shares and leverage side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6']

# Shares
bars = ax1.barh(items, shares, color=colors, edgecolor='white', linewidth=2)
ax1.set_xlabel('Share (ρ)', fontsize=13)
ax1.set_title('Portfolio Shares (weight fraction)', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 0.5)
for bar, s in zip(bars, shares):
    ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{s:.1%}', va='center', fontsize=11)

# Leverage
bars2 = ax2.barh(items, leverage, color=colors, edgecolor='white', linewidth=2)
ax2.set_xlabel('Leverage (1/ρ)', fontsize=13)
ax2.set_title('Leverage (sensitivity to removal)', fontsize=14, fontweight='bold')
ax2.axvline(x=10, color='orange', linestyle='--', alpha=0.7, label='High-leverage threshold')
ax2.legend()
for bar, l in zip(bars2, leverage):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{l:.1f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()
print('Pencil Case has the highest leverage: small share but high sensitivity!')

## 2. The PROOF Line: How Concentrated?

The **PROOF line** counts how many items you need to reach 80% of the portfolio.  
- PROOF = 1 → One item dominates everything (Concentration Trap!)
- PROOF = K → Perfectly spread across all items

Let's compute it.

In [ ]:
# Compute PROOF line
sorted_pairs = sorted(zip(shares, items), reverse=True)
cumsum = 0
proof_line = 0

print('Cumulative share (sorted by size):')
for s, item in sorted_pairs:
    cumsum += s
    proof_line += 1
    marker = ' ← 80% reached!' if cumsum >= 0.8 and (cumsum - s) < 0.8 else ''
    print(f'  {proof_line}. {item:<15} {s:.1%}  (cumulative: {cumsum:.1%}){marker}')

# Reset and compute properly
cumsum = 0
proof_line = 0
for s, item in sorted_pairs:
    cumsum += s
    proof_line += 1
    if cumsum >= 0.8:
        break

print(f'\nPROOF line = {proof_line} out of {len(items)} items')
print(f'  \u2192 You need {proof_line} items to account for 80% of backpack weight.')
if proof_line <= 2:
    print('  \u26a0\ufe0f Portfolio is quite concentrated!')

## 3. What Happens When You Remove an Item?

The unity constraint means: if one share goes down, others MUST go up.  
Let's see what happens when you leave the jacket at home.

In [ ]:
# Remove jacket (index 4)
items_new = ['Books', 'Lunch', 'Water Bottle', 'Pencil Case']
weights_new = [2.0, 1.0, 0.5, 0.3]
total_new = sum(weights_new)
shares_new = [w / total_new for w in weights_new]

print('BEFORE (with jacket):')
for item, s in zip(items, shares):
    print(f'  {item:<15} {s:.1%}')

print(f'\nAFTER (jacket removed):')
for item, s in zip(items_new, shares_new):
    old_s = shares[items.index(item)]
    delta = s - old_s
    print(f'  {item:<15} {s:.1%}  ({delta:+.1%})')

print(f'\n\u2211\u03c1 (before) = {sum(shares):.3f}')
print(f'\u2211\u03c1 (after)  = {sum(shares_new):.3f}')
print('\nUnity constraint holds in BOTH cases!')
print('Every remaining item\'s share went UP \u2014 because the total pot shrank.')
print('This is why shares are relational, not absolute.')

In [ ]:
# Visualize the rebalancing
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(items_new))
width = 0.35

before_shares = [shares[items.index(item)] for item in items_new]
ax.bar(x - width/2, before_shares, width, label='Before (with jacket)', color='#3498DB', alpha=0.8)
ax.bar(x + width/2, shares_new, width, label='After (no jacket)', color='#E74C3C', alpha=0.8)

# Draw arrows showing the shift
for i in range(len(items_new)):
    ax.annotate('', xy=(x[i] + width/2, shares_new[i]),
                xytext=(x[i] - width/2, before_shares[i]),
                arrowprops=dict(arrowstyle='->', color='orange', lw=2))

ax.set_xticks(x)
ax.set_xticklabels(items_new)
ax.set_ylabel('Share (ρ)', fontsize=13)
ax.set_title('Removing the Jacket Rebalances Everything', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()
print('Orange arrows show: removing one element forces ALL shares to change.')
print('This is the simplex at work \u2014 shares are always relational.')

## 4. Try It Yourself!

Model YOUR backpack (or bag, or desk, or fridge) as a portfolio.  
Change the items and weights below and re-run.

In [ ]:
# ✏️ CHANGE THESE to model your own portfolio!
my_items = ['Laptop', 'Charger', 'Notebook', 'Snacks', 'Headphones']
my_weights = [1.5, 0.3, 0.4, 0.2, 0.1]  # kg

# ------- computation -------
my_total = sum(my_weights)
my_shares = [w / my_total for w in my_weights]
my_leverage = [1/s for s in my_shares]

print(f'Portfolio: {my_total:.1f} kg total\n')
print(f'{"Item":<15} {"Weight":<10} {"Share":<10} {"Leverage":<10}')
print('-' * 45)
for item, w, s, l in zip(my_items, my_weights, my_shares, my_leverage):
    flag = ' \u26a0\ufe0f HIGH' if l > 10 else ''
    print(f'{item:<15} {w:<10.1f} {s:<10.1%} {l:<10.1f}{flag}')

# PROOF line
sorted_s = sorted(my_shares, reverse=True)
cum = 0
pl = 0
for s in sorted_s:
    cum += s
    pl += 1
    if cum >= 0.8: break

print(f'\nPROOF line = {pl}/{len(my_items)}')
print(f'Highest leverage: {my_items[my_leverage.index(max(my_leverage))]} ({max(my_leverage):.1f})')

## 5. Summary

| Concept | Backpack Version | HUF Term |
|---------|-----------------|----------|
| Weight limit | Fixed total capacity | **Budget Ceiling** |
| Each item's weight fraction | Proportion of total | **Share (ρᵢ)** |
| How sensitive is removal | 1 / share | **Leverage** |
| Items needed for 80% | Top items by share | **PROOF Line** |
| Removing an item | All shares rebalance | **Simplex constraint** |

**Key takeaway**: Portfolios are *relational*. You can't change one share without  
affecting all the others. HUF tracks these relationships across any domain.

---

### Next Steps

- **Notebook 3**: Real data from sourdough fermentation → `02_sourdough_baker.ipynb`
- **Vol 1**: Formal portfolio theory → *Core Reference: The Unity Constraint*
- **Vol 3**: Why leverage = 1/ρ is mathematically natural → *Mathematical Foundations*